# Generative AI for Data Insights
**Dataset:** [Kaggle — Medical Insurance Price Prediction](https://www.kaggle.com/datasets/harishkumardatalab/medical-insurance-price-prediction)

We ask questions about the data **in plain English**, and an LLM (Claude) answers them by writing SQL, running it on the data, and explaining what it found.

```
question ──► Claude writes SQL ──► run_sql tool executes it ──► Claude reads the result ──► answer + insight
                     ▲                                                  │
                     └──────────── repeats until it has enough ─────────┘
```

**Design choices**
- The model never sees the raw table, only the schema and the query results it asks for.
- The database is **read-only**: Claude can query, never change, the data.
- Every query Claude runs is printed, so each answer can be checked.
- At the end we **verify** one AI answer against plain pandas. Generative AI can be wrong — always check.

## 1. Setup
You need an Anthropic API key from [console.anthropic.com](https://console.anthropic.com). The key is read from the `ANTHROPIC_API_KEY` environment variable; if it is not set, the cell asks for it (it is not saved in the notebook).

In [ ]:
# %pip install anthropic kagglehub   # run once if needed
import os
import sqlite3
from getpass import getpass

import anthropic
import kagglehub
import pandas as pd
from anthropic import beta_tool
from IPython.display import Markdown, display

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")

client = anthropic.Anthropic()
MODEL = "claude-opus-5-5"

## 2. Load the data into a read-only SQL database

In [ ]:
path = kagglehub.dataset_download("harishkumardatalab/medical-insurance-price-prediction")
df = pd.read_csv(os.path.join(path, "Medical_insurance.csv"))
print("Rows in the file:", len(df), "| exact duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)   # half the file is duplicated rows
print("Rows after cleaning:", len(df))

conn = sqlite3.connect(":memory:")
df.to_sql("insurance", conn, index=False)
conn.execute("PRAGMA query_only = ON")   # from here on, the database is read-only
df.head()

## 3. The tool Claude can use
`run_sql` executes one SQL query and returns the result as a small table. Errors are returned as text so Claude can fix its own query.

In [ ]:
MAX_ROWS = 50
query_log = []   # every query Claude runs, for transparency

@beta_tool
def run_sql(query: str) -> str:
    """Run one read-only SQLite query on the `insurance` table and return the result.

    Args:
        query: A single SQLite SELECT statement.
    """
    query_log.append(query)
    try:
        result = pd.read_sql_query(query, conn)
    except Exception as e:
        return f"SQL error: {e}"
    note = f"\n(showing {MAX_ROWS} of {len(result)} rows)" if len(result) > MAX_ROWS else ""
    return result.head(MAX_ROWS).to_csv(index=False, lineterminator="\n") + note

print(run_sql("SELECT COUNT(*) AS n_rows FROM insurance"))   # quick test, no AI involved
query_log.clear()

## 4. The assistant
The system prompt gives Claude the table schema and the rules. The SDK's **tool runner** handles the loop: Claude calls `run_sql`, reads the result, and calls it again until it can answer.

In [ ]:
SCHEMA = """Table `insurance` (1,337 rows, one per insured person):
- age (INTEGER): 18-64
- sex (TEXT): 'male' / 'female'
- bmi (REAL): body mass index
- children (INTEGER): number of dependents covered
- smoker (TEXT): 'yes' / 'no'
- region (TEXT): 'northeast', 'northwest', 'southeast', 'southwest'
- charges (REAL): yearly medical costs billed, in US dollars"""

SYSTEM = f"""You are a data analyst. Answer questions about this SQLite database:

{SCHEMA}

Rules:
- Get every number from the run_sql tool. Never guess or invent a value.
- Prefer a few focused queries over many small ones.
- Answer in short Markdown: the direct answer first, then the key numbers, then one practical insight.
- Say when a finding shows correlation, not causation, or when a group is too small to trust."""


def ask(question: str) -> None:
    """Ask a question in plain English and show the queries Claude ran and its answer."""
    query_log.clear()
    runner = client.beta.messages.tool_runner(
        model=MODEL,
        max_tokens=16000,
        system=SYSTEM,
        tools=[run_sql],
        output_config={"effort": "medium"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",   # if the model declines, the API retries on a fallback model
        messages=[{"role": "user", "content": question}],
    )
    final = runner.until_done()
    for i, q in enumerate(query_log, 1):
        print(f"-- Query {i}\n{q}\n")
    if final.stop_reason == "refusal":
        print("The model declined to answer this question.")
        return
    display(Markdown("".join(b.text for b in final.content if b.type == "text")))

## 5. Ask questions in plain English
### 5.1 A simple count (like the quiz question)

In [ ]:
ask("What is the count of rows in the insurance data set?")

### 5.2 Comparing groups

In [ ]:
ask("How much more do smokers pay than non-smokers on average?")

### 5.3 Combining two factors

In [ ]:
ask("How do charges change with BMI for smokers and non-smokers? Use BMI groups: under 25, 25-30, over 30.")

### 5.4 Open-ended: let the AI find the insights

In [ ]:
ask("You are presenting to an insurance company's pricing team. Find the 3 most important insights in this data.")

### 5.5 Try your own question

In [ ]:
ask("Which region has the highest average charges, and is the difference large?")

## 6. Verify the AI with plain pandas
The AI's numbers must match a calculation we do ourselves. If they don't, the AI answer is wrong.

In [ ]:
check = df.groupby("smoker")["charges"].mean().round(2)
print(check)
print(f"\nSmokers pay {check['yes'] / check['no']:.1f}x more on average "
      f"(${check['yes'] - check['no']:,.0f} more per year).")

## 7. Conclusions
- **Generative AI turns questions into analysis.** You ask in plain English; the model writes the SQL, reads the results, and explains them.
- **Tools keep it grounded.** Claude only answers from query results, and every query is shown, so the work can be audited.
- **Safety by design:** read-only database, the model only sees the results it asks for, and an API key that is never stored in the notebook.
- **Always verify.** LLMs can write a wrong query or misread a result. Check key numbers with your own code (section 6) before sharing them.
- **The data itself:** smoking is by far the strongest driver of charges, and it interacts with BMI — smokers with high BMI pay the most.